# Waterfilling Algorithm Testing and Visualization
This notebook tests the waterfilling algorithm across different network contexts: Opera, Shale, Sirius, and Genetic Algorithm. It includes rigorous testing with Gaussian noise and standardizes power budget to 50 units.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import os
from Waterfilling_Alg import waterfilling
from Shale_Alg import RR2, RR2_path, spray_short
from Common_Alg import generate_random_latin_square
from Sirius import generate_full_system
from AI_Topology import evolve_topology, generate_random_topology, calculate_aspl
from Opera_Alg import find_optimal_path_broken_racks, find_path_2d

# Ensure plots directory exists
if not os.path.exists('plots'):
    os.makedirs('plots')

In [ ]:
def run_rigorous_waterfilling(base_noise, total_power, iterations=500, noise_std=1.0):
    """
    Runs the waterfilling algorithm multiple times with added Gaussian noise 
    and returns the average and standard deviation of allocations.
    """
    base_noise = np.array(base_noise)
    all_allocations = []
    
    for _ in range(iterations):
        # Add random Gaussian noise to the base noise levels
        # Ensuring noise is at least a small positive value
        stochastic_noise = np.maximum(base_noise + np.random.normal(0, noise_std, base_noise.shape), 0.1)
        alloc = waterfilling(stochastic_noise, total_power)
        all_allocations.append(alloc)
        
    return np.mean(all_allocations, axis=0), np.std(all_allocations, axis=0)

def visualize_waterfilling(channels, total_power, title="Waterfilling Results", filename=None, yerr=None):
    """
    Visualizes the waterfilling power allocation using a stacked bar chart.
    Highlights bottlenecks (channels with 0 allocated power due to high noise).
    Supports error bars (yerr) for rigorous testing (averages).
    Saves the image if filename is provided.
    """
    channels = np.array(channels)
    
    # Handle 2D case (multiple timeslots)
    if channels.ndim == 2:
        num_timeslots = channels.shape[0]
        # Handle scalar power vs per-timeslot power
        if np.isscalar(total_power):
            powers = [total_power] * num_timeslots
        else:
            powers = total_power
            
        for t in range(num_timeslots):
            # Recursively call for each timeslot
            ts_suffix = f"_ts{t}"
            ts_filename = f"plots/{filename}{ts_suffix}.png" if filename else None
            ts_yerr = yerr[t] if yerr is not None else None
            visualize_waterfilling(channels[t], powers[t], title=f"{title} - Timeslot {t}", filename=ts_filename, yerr=ts_yerr)
        return

    # --- 1D Logic ---
    allocation = waterfilling(channels, total_power)
    n = len(channels)
    indices = np.arange(n)
    
    active_mask = allocation > 1e-9
    bottleneck_mask = ~active_mask
    
    # Calculate Water Level (Noise + Allocated Power) for active channels
    if np.any(active_mask):
        # All active channels should sum to the same level
        water_levels = channels[active_mask] + allocation[active_mask]
        water_level = np.mean(water_levels) 
    else:
        water_level = 0 

    plt.figure(figsize=(10, 6))
    
    # 1. Plot Active Noise (Gray)
    if np.any(active_mask):
        plt.bar(indices[active_mask], channels[active_mask], 
                label='Noise (Active)', color='lightgray', edgecolor='black')
        
    # 2. Plot Bottleneck Noise (Red/Salmon) - These are the bottlenecks!
    if np.any(bottleneck_mask):
        plt.bar(indices[bottleneck_mask], channels[bottleneck_mask], 
                label='Noise (Bottleneck)', color='salmon', edgecolor='black', hatch='//')

    # 3. Plot Allocated Power (Blue)
    if np.any(active_mask):
        plt.bar(indices[active_mask], allocation[active_mask], bottom=channels[active_mask], 
                label='Allocated Power', color='skyblue', edgecolor='black', yerr=yerr[active_mask] if yerr is not None else None, capsize=5)
    
    # 4. Water Level Line
    plt.axhline(y=water_level, color='blue', linestyle='--', linewidth=2, label=f'Water Level ({water_level:.2f})')
    
    plt.xlabel('Channel / Link Index')
    plt.ylabel('Power / Noise Level')
    plt.title(title)
    plt.xticks(indices, [f'Ch {i}' for i in indices])
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    if filename:
        save_path = filename if filename.startswith('plots/') else f"plots/{filename}"
        if not save_path.endswith('.png'):
            save_path += '.png'
        plt.savefig(save_path)
        print(f"Saved plot to {save_path}")
    plt.show()

## 1. Opera Context
Testing waterfilling for rack allocations in the Opera architecture.

In [ ]:
print("\n=== Testing Waterfilling with Opera Context ===")
rack_weights = [5, 10, 5, 20, 100, 5, 10, 5]
total_power = 50
allocation = waterfilling(rack_weights, total_power)

visualize_waterfilling(rack_weights, total_power, title="Opera Waterfilling Bottlenecks")

print("\n--- Rigorous Test (Averaged over 500 iterations with noise) ---")
avg_p, std_p = run_rigorous_waterfilling(rack_weights, total_power)
visualize_waterfilling(rack_weights, total_power, title="Opera Average Allocation (Rigorous)", filename="opera_rigorous", yerr=std_p)

## 2. Shale Context
Testing waterfilling for link power allocation in a Shale topology (RR2).

In [ ]:
print("\n=== Testing Waterfilling with Shale Context ===")
adj_matrix = RR2(3, 2)
active_links = [x for x in adj_matrix[0] if x is not None]
num_links = len(active_links)

random.seed(42)
link_noise = [random.randint(1, 20) for _ in range(num_links)]
total_power = 50

visualize_waterfilling(link_noise, total_power, title="Shale Waterfilling Bottlenecks")

print("\n--- Rigorous Test (Averaged over 500 iterations with noise) ---")
avg_p, std_p = run_rigorous_waterfilling(link_noise, total_power)
visualize_waterfilling(link_noise, total_power, title="Shale Average Allocation (Rigorous)", filename="shale_rigorous", yerr=std_p)

## 3. Sirius Context (Multi-Timeslot)
Testing time-slotted power allocation in the Sirius architecture.

In [ ]:
print("\n=== Testing Waterfilling with Sirius Context ===")
wavelengths, ports, nodes = 3, 2, 6
As, Ws, P = generate_full_system(wavelengths, ports, nodes)

channels = []
random.seed(100)
for t, W in enumerate(Ws):
    timeslot_noise = [random.randint(1, 20) for _ in range(nodes)]
    channels.append(timeslot_noise)

total_power = 50
visualize_waterfilling(channels, total_power, title="Sirius Waterfilling Bottlenecks")

print("\n--- Rigorous Test (Averaged per Timeslot) ---")
sirius_std_all = []
for t in range(len(Ws)):
    avg_p, std_p = run_rigorous_waterfilling(channels[t], total_power)
    sirius_std_all.append(std_p)

visualize_waterfilling(channels, total_power, title="Sirius Averaged (Rigorous)", filename="sirius_rigorous", yerr=np.array(sirius_std_all))

## 4. Diurnal Traffic Patterns
Modeling power budget based on a day/night cycle.

In [ ]:
def generate_diurnal_power(num_timeslots, baseline=30, amplitude=20):
    t = np.arange(num_timeslots)
    return baseline + amplitude * np.sin(2 * np.pi * t / num_timeslots)

num_timeslots, num_channels = 24, 5
noise = [10, 15, 5, 20, 25]
channels = [noise for _ in range(num_timeslots)]
power_budgets = generate_diurnal_power(num_timeslots)
allocations = waterfilling(channels, power_budgets)

capacities = [np.sum(np.log2(1 + allocations[t]/np.array(noise))) for t in range(num_timeslots)]

plt.figure(figsize=(10, 5))
plt.plot(range(num_timeslots), power_budgets, label='Power Budget (Diurnal)', color='orange', marker='o')
plt.plot(range(num_timeslots), capacities, label='Capacity (Sum Rate)', color='blue', marker='x')
plt.title("Diurnal Waterfilling Performance")
plt.legend()
plt.show()

visualize_waterfilling(noise, power_budgets[12], title="Diurnal - Noon (Peak)", filename="diurnal_noon")
visualize_waterfilling(noise, power_budgets[0], title="Diurnal - Midnight (Low)", filename="diurnal_midnight")

## 5. Latency Scenarios
Comparing power allocation under low vs. high latency (noise accumulation) conditions.

In [ ]:
noise = [random.randint(5, 15) for _ in range(8)]
print("Opera Low vs High Latency")
visualize_waterfilling(noise, 50, title="Opera - Low Latency", filename="opera_low_latency")
visualize_waterfilling([n * 3 for n in noise], 50, title="Opera - High Latency", filename="opera_high_latency")

shale_noise = [5, 2, 8, 4]
print("Shale Low vs High Latency")
visualize_waterfilling(shale_noise, 50, title="Shale - Low Latency", filename="shale_low_latency")
visualize_waterfilling([n * 1.5 for n in shale_noise], 50, title="Shale - High Latency", filename="shale_high_latency")

## 6. Genetic Algorithm Optimized Topologies
Performing waterfilling on links of evolved topologies.

In [ ]:
num_nodes, degree = 6, 2
best_adj = evolve_topology(num_nodes, degree, population_size=10, generations=20)
total_power = 50

for i in range(num_nodes):
    neighbors = best_adj[i]
    node_noise = [random.randint(5, 25) for _ in range(len(neighbors))]
    # Standard Test
    visualize_waterfilling(node_noise, total_power, title=f"Genetic Algo - Node {i} Neighborhood", filename=f"genetic_node_{i}")
    # Rigorous Test
    avg_p, std_p = run_rigorous_waterfilling(node_noise, total_power)
    visualize_waterfilling(node_noise, total_power, title=f"Genetic Algo - Node {i} Rigorous", filename=f"genetic_node_{i}_rigorous", yerr=std_p)

## 7. Performance Comparison
Comparing Shannon Capacity across all contexts as power budget increases.

In [ ]:
power_levels = np.linspace(10, 100, 10)
scenarios = [
    ("Opera", [5, 10, 5, 20, 100, 5, 10, 5]),
    ("Shale", [4, 1, 9, 8]),
    ("Sirius", [5, 15, 15, 6, 13, 12]),
    ("Generic", [5, 5, 5, 5])
]

plt.figure(figsize=(10, 6))
for label, noises in scenarios:
    capacities = [np.sum(np.log2(1 + waterfilling(noises, P)/np.array(noises))) for P in power_levels]
    plt.plot(power_levels, capacities, marker='o', label=label)

plt.xlabel("Total Power Budget (P)")
plt.ylabel("Capacity (Shannon Sum Rate)")
plt.title("Waterfilling Performance Comparison")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()